![Alt text](ccats_logo.png)
# Quantum-electrodynamical time-dependent density functional theory within Gaussian atomic basis - A tutorial for TDA-JC, RWA, Rabi, and PF

#### **Reference:** https://doi.org/10.1063/5.0057542
---
## Overview
This tutorial examines the effect of the Tamm-Dancoff approximation with Jaynes-Cummings (TDA-JC), rotating-wave approximation (RWA), Rabi, and Pauli-Fierz (PF) models to model the polariton spectrum of molecules in optical cavities.

This tutorial is divided into the following parts: 

1. **Part 1 -** Configuration and TDA-JC Basics
2. **Part 2 -** Polariton spectrum as a function of coupling strength (Comparing JC, RWA, Rabi, and PF)
3. **Part 3 -** TDA-RWA application 
4. **Part 4 -** TDA-Rabi & TDA-PF comparison
5. **Part 5 -** Capstone Exercise

---
## Environment Setup

### Option A: Running in Google Colab
If you opened this notebook in Google Colab, the environment configuration is handled automatically. 
1. Run the **Setup Cell** below. It will clone the repository, setup the directory structure, and install all dependencies.
2. Proceed to the **Verify Installation** section.

In [ ]:
import sys
import os

if 'google.colab' in sys.modules:
    print("Setting up Colab environment...")
    
    # Clone the repository to access local modules and data files
    !git clone https://github.com/cc-ats/molecular-polariton-modeling.git
    
    # Move into the repository directory
    %cd molecular-polariton-modeling
    
    # Install required packages
    !pip install -r requirements.txt
    
    print("Colab environment set up successfully!")
else:
    print("Running locally. No setup needed.")

### Option B: Running Locally
If you are running this notebook locally (e.g., in VS Code or JupyterLab), follow these steps to configure your enviroment.

#### 1. (Optional) Create a Virtual Environment
It is recommended to use a virtual environment to avoid package conflicts. 

In, your terminal, run:

```bash
python3 -m venv venv
```
Next, activate the virtual environment with one of these commands.

MacOS:

```bash
source venv/bin/activate
```

Windows:

```bash
.venv\Scripts\activate
```

#### 2. Install Dependencies
Install all required Python packages using:

```bash
pip install -r requirements.txt
```

#### 3. Select the Jupyter Kernel
In the top-right corner of the notebook: 
1. Click **Kernel**
2. Select **Change Kernel**
3. Choose the `.venv` environment you just created

#### 4. Run the following command in your terminal to setup the directories
```bash
export PYTHONPATH=$(pwd):$PYTHONPATH
```

### Verify Installation
Run the following cell to import all required libraries and confirm that the environment is configured correctly.

### 📦 Imports & Configuration

In [ ]:
import numpy as np
from pyscf import gto, scf, tdscf
import qed
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

# Color for printing
RED    = "\033[91m"
GREEN  = "\033[92m"
YELLOW = "\033[93m"
BLUE   = "\033[94m"
RESET  = "\033[0m"

#Unit Conversion:
HF_TO_EV = 27.2114
EV_TO_HF = 1 / HF_TO_EV

print("✅ All the imports are successful!")

--- 
# 🗂️ Part 1: Configuration
## 1.1 Molecular Structure and Ground State Calculation
We can specify the structure and basis of the molecule directly in the follow code block or any structural file. Then, we run the ground state SCF calculation:

In [ ]:
mol_file = 'ethene.xyz'  # Replace with your molecule file path
mol = gto.M(atom=mol_file, basis='6-311++G')
mf = scf.RKS(mol)
mf.xc = 'pbe0'
mf.kernel();

## 1.2 Theoretical Foundations

Under the Tamm-Dancoff approximation (TDA), the de-excitation block in the electronic part is set to zero. Depending on which terms are kept in the light-matter coupling Hamiltonian, we obtain four distinct models:

### 1. The Tamm-Dancoff Pauli-Fierz (TDA-PF) Model
The most physically complete ab initio model, including the dipole self-energy (DSE) $\mathbf{\Delta}$ and all counter-rotating terms (CRTs). The polaritonic states are found by solving the following non-Hermitian eigenvalue problem:

$$
\begin{bmatrix}
\mathbf{A} + \mathbf{\Delta} & \hbar \mathbf{g}^{\dagger} & \hbar \tilde{\mathbf{g}}^{\dagger} \\
\hbar \mathbf{g} & \hbar \mathbf{\omega} & \mathbf{0} \\
\hbar \tilde{\mathbf{g}} & \mathbf{0} & \hbar \mathbf{\omega}
\end{bmatrix}
\begin{bmatrix}
\mathbf{X} \\ \mathbf{M} \\ \mathbf{N}
\end{bmatrix}
=
\hbar\Omega^{\text{TDA-PF}}
\begin{bmatrix}
\mathbf{1} & \mathbf{0} & \mathbf{0} \\
\mathbf{0} & \mathbf{1} & \mathbf{0} \\
\mathbf{0} & \mathbf{0} & -\mathbf{1} 
\end{bmatrix}
\begin{bmatrix}
\mathbf{X} \\ \mathbf{M} \\ \mathbf{N}
\end{bmatrix}
\tag{1}
$$

where: 
*   $\mathbf{A}$ is the standard TDDFT electronic excitation energy matrix (TDA block).
*   $\hbar\mathbf{\omega}$ is a diagonal matrix containing the cavity photon mode frequencies.
*   $\mathbf{X}$ is the electronic transition amplitude vector.
*   $\mathbf{M}$ and $\mathbf{N}$ are the transition amplitudes for the photon annihilation (absorption) and photon creation (emission) operators, respectively.
*   $\mathbf{g}$ and $\tilde{\mathbf{g}}$ are the electron-photon coupling strength matrices. For standard dipole coupling, they are equal: $\mathbf{g} = \tilde{\mathbf{g}} = \mathbf{g}_{\text{coup}}$, where each element is:
    $$g_{ia} = \sqrt{\frac{\omega_c}{2}}(\boldsymbol{\lambda}\cdot\boldsymbol{\mu}_{ia})$$
    with $\omega_c$ being the cavity frequency, $\boldsymbol{\lambda}$ the vacuum field coupling vector, and $\boldsymbol{\mu}_{ia}$ the transition dipole moment between occupied orbital $i$ and virtual orbital $a$.
*   $\mathbf{\Delta}$ is the Dipole Self-Energy (DSE) matrix:
    $$\Delta_{ia, jb} = \frac{1}{2}(\boldsymbol{\lambda}\cdot\boldsymbol{\mu}_{ia})(\boldsymbol{\lambda}\cdot\boldsymbol{\mu}_{jb})$$
    which represents the harmonic potential felt by the photon field due to the molecular dipole.

### 2. The TDA-Rabi Model
If we neglect the dipole self-energy (DSE) matrix ($\mathbf{\Delta} = \mathbf{0}$) but retain the counter-rotating terms (CRTs) $\tilde{\mathbf{g}}$, we obtain the TDA-Rabi model:

$$
\begin{bmatrix}
\mathbf{A} & \hbar \mathbf{g}^{\dagger} & \hbar \tilde{\mathbf{g}}^{\dagger} \\
\hbar \mathbf{g} & \hbar \mathbf{\omega} & \mathbf{0} \\
\hbar \tilde{\mathbf{g}} & \mathbf{0} & \hbar \mathbf{\omega}
\end{bmatrix}
\begin{bmatrix}
\mathbf{X} \\ \mathbf{M} \\ \mathbf{N}
\end{bmatrix}
=
\hbar\Omega^{\text{TDA-Rabi}}
\begin{bmatrix}
\mathbf{1} & \mathbf{0} & \mathbf{0} \\
\mathbf{0} & \mathbf{1} & \mathbf{0} \\
\mathbf{0} & \mathbf{0} & -\mathbf{1} 
\end{bmatrix}
\begin{bmatrix}
\mathbf{X} \\ \mathbf{M} \\ \mathbf{N}
\end{bmatrix}
\tag{2}
$$

### 3. The TDA Rotating-Wave Approximation (TDA-RWA) Model
By neglecting the counter-rotating terms (CRTs), we set the photon emission amplitude $\mathbf{N} = \mathbf{0}$. If we keep the DSE matrix $\mathbf{\Delta}$, we get the TDA-RWA model:

$$
\begin{bmatrix}
\mathbf{A} + \mathbf{\Delta} & \hbar \mathbf{g}^{\dagger}\\
\hbar \mathbf{g} & \hbar \mathbf{\omega}
\end{bmatrix}
\begin{bmatrix}
\mathbf{X} \\ \mathbf{M}
\end{bmatrix}
=
\hbar\Omega^{\text{TDA-RWA}}
\begin{bmatrix}
\mathbf{1} & \mathbf{0}\\
\mathbf{0} & \mathbf{1}
\end{bmatrix}
\begin{bmatrix}
\mathbf{X} \\ \mathbf{M}
\end{bmatrix}
\tag{3}
$$

### 4. The TDA Jaynes-Cummings (TDA-JC) Model
If we neglect both the DSE ($\mathbf{\Delta} = \mathbf{0}$) and the CRTs ($\mathbf{N} = \mathbf{0}$), we get the standard TDA-JC model:

$$
\begin{bmatrix}
\mathbf{A} & \hbar \mathbf{g}^{\dagger}\\
\hbar \mathbf{g} & \hbar \mathbf{\omega}
\end{bmatrix}
\begin{bmatrix}
\mathbf{X} \\ \mathbf{M}
\end{bmatrix}
=
\hbar\Omega^{\text{TDA-JC}}
\begin{bmatrix}
\mathbf{1} & \mathbf{0}\\
\mathbf{0} & \mathbf{1}
\end{bmatrix}
\begin{bmatrix}
\mathbf{X} \\ \mathbf{M}
\end{bmatrix}
\tag{4}
$$

---

### 💡 Physical Intuition & Coupling Regimes

To choose and interpret these models, it is essential to understand the physical coupling regimes, which depend on the coupling strength $\lambda = \|\boldsymbol{\lambda}\|$ relative to the cavity frequency $\omega_c$:

1.  **Weak Coupling:** The rate of coherent light-matter energy exchange is slower than the dissipation rates (cavity loss $\kappa$ and molecular decay $\gamma$).
2.  **Strong Coupling (SC):** Coherent coupling exceeds the decay rates ($g > \kappa, \gamma$). The degenerated exciton-photon system splits into **Lower Polariton (LP)** and **Upper Polariton (UP)** states, separated by the *Rabi splitting* $\Omega_R \approx 2g$.
3.  **Ultra-Strong Coupling (USC):** The coupling strength becomes a significant fraction of the cavity/electronic transition frequency ($\lambda/\omega_c \gtrsim 0.1$). In this regime:
    *   **Rotating Wave Approximation (RWA) Breaks Down:** CRTs (virtual absorption/emission processes) cannot be neglected because they significantly shift polariton energies.
    *   **Dipole Self-Energy (DSE) is Crucial:** Omitting DSE ($_\mathbf{\Delta}$) causes a catastrophic unphysical "ground-state collapse" where the LP energy drops without a lower bound. Mathematically, $_\mathbf{\Delta}$ is required to guarantee gauge invariance and maintain spectral stability.
4.  **Deep Strong Coupling (DSC):** The coupling strength is comparable to or exceeds the transition frequency ($\lambda/\omega_c \ge 1$).

*   **Symplectic / Non-Hermitian Metric:** The metric matrix $\begin{bmatrix} \mathbf{1} & \mathbf{0} & \mathbf{0} \\ \mathbf{0} & \mathbf{1} & \mathbf{0} \\ \mathbf{0} & \mathbf{0} & -\mathbf{1} \end{bmatrix}$ in the PF and Rabi equations is a consequence of the bosonic commutation relations for the cavity photon operators ($[a, a^\dagger] = 1$). Under RWA/JC, neglecting the creation amplitude ($\mathbf{N}=\mathbf{0}$) removes the negative metric block, transforming the system into a standard Hermitian eigenvalue problem.

## 1.3 Computational Workflow
### 1.3.1 Calculate Transition Dipole
1. We first need to construct the excited state solver on top of the mean-field object reference and obtain the excitation energies from the TDA eigenvalue equation.
2. Next, we compute the oscillator strengths, which measure how strongly light couples to a transition:
$$
f_I = \frac{2}{3}\omega_I|\mu_{0I}|^2
$$
3. Then, we get the largest oscillator strength to identify the brightest electronic state and compute its transition dipole moment, which is defined as:
$$
\mathbf{\mu}_I^{\text{TDA}}=\sum_{ai}X^{\text{TDA}}_{I,ai}\mathbf{\mu}_{ai}
$$

**Note on Oscillator Strength:** The `oscillator_strength()` represents the dimensionless probability of a molecule interacting with electromagnetic radiation to undergo a specific electronic transition. A higher value indicates a brighter state. In cavity QED simulations, we typically align the cavity field polarization ($\vec{\epsilon}$) with the transition dipole moment ($\vec{d}_{ig}$) of the brightest state to achieve the maximum possible light-matter coupling strength.

In [ ]:
td = tdscf.TDA(mf)
td.nroots = 5 # Request 5 exicted states
td.kernel()

osc = td.oscillator_strength() # Compute oscillator strengths for each excitation
bright_idx = np.argmax(osc)
target_energy = td.e[bright_idx]

# Get transition dipole moment for the brightest excitation
trans_dip = td.transition_dipole()[bright_idx]
print(f"Targeting bright state: {bright_idx}, at {target_energy:.4f} a.u.")
print(f"Transition dipole moment (a.u.): {trans_dip}")

Next, we need to solve the full ab initio QED-TDDFT equations in the molecular orbital basis (Equation 4) 

### 1.3.2 Setup QED-TDA with JC Model

The TDA-JC method from ```qed``` package require a key which can contains:
| Category | Key | Type | Default | Description |
| --- | --- | --- | --- | --- |
| **Physical** | `cavity_freq` | float / array | **Required** | Photon mode energy in atomic units (Hartree). |
| | `cavity_mode` | array (3, N) | **Required** | Vector(s) defining coupling direction and vacuum field strength $\mathbf{\lambda}$. |
| | `cavity_model` | string | `'JC'` | Choice of Hamiltonian: `'JC'`, `'RWA'`, `'Rabi'`, or `'PF'`. |
| | `uniform_field` | boolean | `True` | Whether the vacuum field is spatially uniform across the molecule. |
| **Solver** | `nstates` | integer | `4` | Total number of **polaritonic** roots (eigenvalues) to solve for. |
| | `target_states` | string | `'polariton'` | Type of states to extract (e.g., `'polariton'`, `'exciton'`). |
| | `solver_algorithm`| string | `'davidson_qr'`| Algorithm: `'davidson_qr'` (ab initio) or `'direct'` (FewLevel). |
| | `tolerance` | float | `1e-8` | Convergence threshold for the eigenvalue solver residual. |
| | `max_cycle` | integer | `100` | Maximum iterations allowed for iterative solvers. |
| | `level_shift` | float | `0.0` | Numerical shift applied to diagonal elements to aid convergence. |
| **Tuning** | `resonance_state`| integer | `None` | Index of the TDA electronic state to which the photon should be tuned. |
| | `adjust_func` | string | `None` | Tuning strategy for frequency (e.g., `'average'`). Requires `resonance_state`.|
| **Collective**| `has_offdiag` | boolean | `False` | Include inter-fragment dipole-dipole interactions between molecules. |
| | `scale_coupling` | float | `0.0` | Scaling factor for coupling strength, typically $1/\sqrt{N_{frag}}$. |

In [ ]:
# Calculate unit vector of the transition dipole moment
unit_dip = trans_dip / np.linalg.norm(trans_dip)

#Define Cavity Parameters with lambda = 0.1 a.u. (can be change) and coupling direction along the transition dipole moment 
lambda_coupling = 0.1  # Coupling strength in atomic units
cavity_mode = (unit_dip * lambda_coupling).reshape(3, 1)  # Cavity mode vector along the transition dipole direction

key = {
    'cavity_mode': cavity_mode,
    'cavity_freq': np.array([target_energy]),
}

# Construct TDA-JC Model with qed package
cav_model = qed.JC(mf, key)
qed_td = qed.TDA(mf, td, cav_model, key)

# Run Calculation
qed_td.nroots = 8
qed_td.kernel();

print(f"\n{GREEN}Polaritonic states computed successfully!{RESET}")
print(f"Polaritonic energies (in eV): {qed_td.e * HF_TO_EV}")
print(f"Transition dipole moments (a.u.):")
print(qed_td.trans_dip)

### 1.3.2 🏋️ **Exercise: The Effect of Detuning**
Here, we tuned the cavity frequency exactly to the electronic transition ($\delta = 0$).
*   **Task:** In Part 1.3.2, manually set `cavity_freq` to be 0.5 eV higher than the `target_energy`.
    *(Hint: You will need to convert 0.5 eV to Hartree using `0.5 * EV_TO_HF` and add it to `target_energy` before setting `cavity_freq` in the `key` dictionary.)*
*   **Question:** Look at the resulting polariton energies and the photon contribution. How does "detuning" the cavity affect the mixing between the exciton and the photon? Does one state become "more photonic" than the other?

---
# 🔬 Part 2: Scanning Coupling Strength ($\lambda$)
Before we scan the coupling strength, it is crucial to define our parameters physically. The fundamental coupling strength ($\lambda$) is expressed in atomic units ($e^{-1} a_0^{-1/2}$). 

Physically, increasing $\lambda$ corresponds to a tighter confinement of the electromagnetic field—essentially decreasing the effective mode volume of the optical cavity. This tighter spatial confinement leads to a stronger, more entangled interaction between the vacuum photon field and the molecule's transition dipole moment. 

To ensure maximum resonance, we will dynamically align the cavity polarization vector to perfectly match the molecule's transition dipole unit vector at every step.

In [ ]:
lambdas = np.linspace(0.0 , 0.10, 12)
cavity_freqs = np.array([target_energy])

JC_energies = []
JC_photon_contribution = [] 

for lam in lambdas:
    print(f"{GREEN}Coupling lam = {lam:.3f} au...{RESET}", end="\r")
    # 1. Update mode adn re-run TDA-JC
    unit_dip = trans_dip / np.linalg.norm(trans_dip)
    cavity_mode = (unit_dip * lam).reshape(3, 1)
    key = {'cavity_mode': cavity_mode, 'cavity_freq': cavity_freqs}
    cav_jc = qed.JC(mf, key)
    td_jc = qed.TDA(mf, td, cav_jc, key)
    td_jc.nroots = 5
    td_jc.kernel()
    print(f"Coupling: {lam:.3f} a.u., Polariton Energies: {td_jc.e}")
    # 2. Extract energies
    JC_energies.append(td_jc.e)

    # 3. Extract photon contribution
    p = cav_jc.get_mns_weight(td_jc.mn)
    JC_photon_contribution.append(p.flatten())
    print(f"{YELLOW}----------------------------------------------------------------{RESET}")

JC_energies = np.array(JC_energies)
JC_photon_contribution = np.array(JC_photon_contribution)

## 💡 Generate Plot

+ Since the cavity frequency is tuned excactly to the brightest electronic transition, the photon mode and the electronic state are **degenerate** at $\lambda = 0$
+ When $\lambda > 0$, this degenerate pair splits into the Lower Polariton (Root 0) and Upper Polariton (Root 1)
+ Root 2 is the next electronic state

In [ ]:
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))

# Energy Plot
ax1.plot(lambdas, JC_energies[:, 0] * HF_TO_EV, 'ro-', markersize=6, alpha=0.7, label='Lower Polariton (LP)')
ax1.plot(lambdas, JC_energies[:, 1] * HF_TO_EV, 'bo-', markersize=6, alpha=0.7, label='Upper Polariton (UP)')
ax1.plot(lambdas, JC_energies[:, 2] * HF_TO_EV, 'go-', markersize=6, alpha=0.7, label='Third Eigenstate')
ax1.axhline(target_energy * HF_TO_EV, color='gray', linestyle='--', label='Resonant Frequency')
ax1.axvline(0, color='black', linewidth=1, linestyle='-', alpha=0.5)

# Focus on the splitting area
y_min = np.min(JC_energies[:, :2] * HF_TO_EV) - 0.5
y_max = np.max(JC_energies[:, :2] * HF_TO_EV) + 1.0
ax1.set_ylim(y_min, y_max)

ax1.set_xlabel('Coupling Strength $\lambda$ (a.u.)')
ax1.set_ylabel('Energy (eV)')
ax1.set_title('Polariton Energies vs Coupling')
ax1.legend()
ax1.grid(alpha=0.3)

# Photon Contribution Plot
mask = lambdas >= 1e-8
ax2.plot(lambdas[mask], JC_photon_contribution[:, 0][mask], 'ro-', markersize=6, alpha=0.7, label='LP Photon Character')
ax2.plot(lambdas[mask], JC_photon_contribution[:, 1][mask], 'bo-', markersize=6, alpha=0.7, label='UP Photon Character')
ax2.axhline(0.5, color='black', linestyle=':', alpha=0.6, label='50% Photon/Exciton')
ax2.axvline(0, color='black', linewidth=1, linestyle='-', alpha=0.5)

ax2.set_xlabel('Coupling Strength $\lambda$ (a.u.)')
ax2.set_ylabel('Photon Contribution')
ax2.set_title('Hybrid Nature of Polaritons')
ax2.legend()
ax2.grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 🏋️ Exercise: Exploring a New Molecule (Formaldehyde)
Apply the tutorial workflow to a different molecular system.
*   **Task:** Replace the `ethene.xyz` geometry with Formaldehyde. You can define it directly in the code:
    ```python
    mol_geometry = """
    C   0.0000000   0.0000000   0.0000000
    O   0.0000000   0.0000000   1.2200000
    H   0.0000000   0.9400000  -0.5800000
    H   0.0000000  -0.9400000  -0.5800000
    """
    mol = gto.M(atom=mol_geometry, basis='6-311++G')
    ```
*   **Question:** Identify the brightest state for Formaldehyde. Is it at a higher or lower energy than Ethene?
    *(Reference Answer for self-check: The brightest transition of Formaldehyde is the 3rd excited state, corresponding to index 2, with an energy of $\approx 7.80$ eV, which is lower than Ethene's brightest transition at $\approx 9.10$ eV.)*

---
# Part 3: Tamm-Dancoff with Rotating Wave Approximation (TDA-RWA)

## 3.1 Computational Workflow
Equation 3 is what we need to solve for this method. As such, we can follow the same structure as TDA-JC, but replace the JC attribute with RWA.

In [ ]:
cav_rwa = qed.RWA(mf, key)
qed_rwa = qed.TDA(mf, td, cav_rwa, key)

qed_rwa.nroots = 8
qed_rwa.kernel()

## 3.2 Compare TDA-JC vs TDA-RWA

In [ ]:
RWA_energies = []
RWA_photon_contribution = []

for lam in lambdas:
    print(f"{GREEN}Coupling lam = {lam:.3f} au...{RESET}", end="\r")
    key['cavity_mode'] = (unit_dip * lam).reshape(3, 1)
    cav_rwa = qed.RWA(mf, key)
    td_rwa = qed.TDA(mf, td, cav_rwa, key)
    td_rwa.nroots = 5
    td_rwa.kernel()
    print(f"Coupling: {lam:.3f} a.u., Polariton Energies: {td_rwa.e}")
    p = cav_rwa.get_mns_weight(td_rwa.mn)
    RWA_photon_contribution.append(p.flatten())
    RWA_energies.append(td_rwa.e)
    print(f"{YELLOW}----------------------------------------------------------------{RESET}")
    
RWA_energies = np.array(RWA_energies)
RWA_photon_contribution = np.array(RWA_photon_contribution)

fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(10, 6))

ax1.plot(lambdas, JC_energies[:, 0] * HF_TO_EV, color='tab:red', marker='o', linestyle='-', markersize=6, alpha=0.7, label='JC Lower Polariton')
ax1.plot(lambdas, JC_energies[:, 1] * HF_TO_EV, color='tab:blue', marker='o', linestyle='-', markersize=6, alpha=0.7, label='JC Upper Polariton')
ax1.plot(lambdas, JC_energies[:, 2] * HF_TO_EV, color='tab:green', marker='o', linestyle='-', markersize=6, alpha=0.7, label='JC Third Eigenstate')
ax1.plot(lambdas, RWA_energies[:, 0] * HF_TO_EV, color='tab:orange', marker='s', linestyle='--', markersize=6, alpha=0.7, label='RWA Lower Polariton')
ax1.plot(lambdas, RWA_energies[:, 1] * HF_TO_EV, color='tab:purple', marker='s', linestyle='--', markersize=6, alpha=0.7, label='RWA Upper Polariton')
ax1.plot(lambdas, RWA_energies[:, 2] * HF_TO_EV, color='tab:brown', marker='s', linestyle='--', markersize=6, alpha=0.7, label='RWA Third Eigenstate')
ax1.axhline(target_energy * HF_TO_EV, color='gray', linestyle='--', label='Underlying Bright State')

ax1.set_xlabel('Coupling Strength (a.u.)')
ax1.set_ylabel('Energy (eV)')
ax1.set_title('Polariton Energies vs Coupling Strength')
ax1.legend()
ax1.grid(alpha=0.3)

mask = lambdas >= 1e-8

ax2.plot(lambdas[mask], JC_photon_contribution[:, 0][mask], color='tab:red', marker='o', linestyle='-', markersize=6, alpha=0.7, label='JC Lower Polariton')
ax2.plot(lambdas[mask], JC_photon_contribution[:, 1][mask], color='tab:blue', marker='o', linestyle='-', markersize=6, alpha=0.7, label='JC Upper Polariton')
ax2.plot(lambdas[mask], JC_photon_contribution[:, 2][mask], color='tab:green', marker='o', linestyle='-', markersize=6, alpha=0.7, label='JC Third Polariton')
ax2.plot(lambdas[mask], RWA_photon_contribution[:, 0][mask], color='tab:orange', marker='s', linestyle='--', markersize=6, alpha=0.7, label='RWA Lower Polariton')
ax2.plot(lambdas[mask], RWA_photon_contribution[:, 1][mask], color='tab:purple', marker='s', linestyle='--', markersize=6, alpha=0.7, label='RWA Upper Polariton')
ax2.plot(lambdas[mask], RWA_photon_contribution[:, 2][mask], color='tab:brown', marker='s', linestyle='--', markersize=6, alpha=0.7, label='RWA Third Polariton')

ax2.set_xlabel('Coupling Strength (a.u.)')
ax2.set_ylabel('Photon Contribution')
ax2.set_title('Photon Contribution vs Coupling Strength')
ax2.legend()
ax2.grid(alpha=0.3)

fig.tight_layout()
plt.show()

---
# Part 4: Tamm-Dancoff approximation with Rabi model
## 4.1 Computational Workflow
The key equation for this is Equation 2. To implement this, we follow the same procedure as before:

In [ ]:
cav_rabi = qed.Rabi(mf, key) # Using Rabi model
qed_rabi = qed.TDA(mf, td, cav_rabi, key)

qed_rabi.nroots = 8
qed_rabi.kernel()

## 4.2 Compare TDA-JC, TDA-RWA, and TDA-Rabi

**Important note:** JC/RWA uses a **single** photon amplitude $M$ so that the photon weight is simply $M^2$. However, Rabi/PF uses **two** photon amplitudes: $M$ (associated with photon annihilation/absorption) and $N$ (associated with photon creation/emission). Because of the non-Hermitian bosonic metric, the polaritonic wavefunction normalization under TDA is:
$$\mathbf{X}^\dagger \mathbf{X} + \mathbf{M}^\dagger \mathbf{M} - \mathbf{N}^\dagger \mathbf{N} = 1$$
The actual "photon weight" (representing the net photon number expectation value) is therefore defined as:
$$W_{ph} = M^2 - N^2$$

In [ ]:
Rabi_energies = []
Rabi_photon_contribution = []
PF_energies = []
PF_photon_contribution = []

for lam in lambdas:
    print(f"{BLUE}Coupling lam = {lam:.3f}, au...{RESET}", end="\r")
    key['cavity_mode'] = (unit_dip * lam).reshape(3, 1)
    
    # 1. Rabi
    cav_rabi = qed.Rabi(mf, key)
    td_rabi = qed.TDA(mf, td, cav_rabi, key)
    td_rabi.nroots = 5
    td_rabi.kernel()
    p_rabi = cav_rabi.get_mns_weight(td_rabi.mn)
    Rabi_photon_contribution.append(p_rabi)
    Rabi_energies.append(td_rabi.e)
    
    # 2. PF
    cav_pf = qed.PF(mf, key)
    td_pf = qed.TDA(mf, td, cav_pf, key)
    td_pf.nroots = 5
    td_pf.kernel()
    p_pf = cav_pf.get_mns_weight(td_pf.mn)
    PF_photon_contribution.append(p_pf)
    PF_energies.append(td_pf.e)
    
    print(f"Coupling: {lam:.3f} a.u., Polariton Energies: {td_rabi.e}")
    print(f"{YELLOW}----------------------------------------------------------------{RESET}")

Rabi_energies = np.array(Rabi_energies)
Rabi_photon_contribution = np.array(Rabi_photon_contribution)
PF_energies = np.array(PF_energies)
PF_photon_contribution = np.array(PF_photon_contribution)

fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(10, 10))

state_colors = ['tab:red', 'tab:blue', 'tab:green']

# JC
for i, label in enumerate([
    'Lower Polariton',
    'Upper Polariton',
    'Third Eigenstate'
]):
    ax1.plot(
        lambdas,
        JC_energies[:, i] * HF_TO_EV, color=state_colors[i], marker='o', linestyle='-',
        linewidth=2,
        markersize=6,
        alpha=0.85,
        label=f'JC {label}'
    )

# RWA
for i, label in enumerate([
    'Lower Polariton',
    'Upper Polariton',
    'Third Eigenstate'
]):
    ax1.plot(
        lambdas,
        RWA_energies[:, i] * HF_TO_EV, color=state_colors[i], marker='s', linestyle='--',
        linewidth=2,
        markersize=6,
        alpha=0.85,
        label=f'RWA {label}'
    )

# Rabi
for i, label in enumerate([
    'Lower Polariton',
    'Upper Polariton',
    'Third Eigenstate'
]):
    ax1.plot(
        lambdas,
        Rabi_energies[:, i] * HF_TO_EV, color=state_colors[i], marker='^', linestyle=':',
        linewidth=2.5,
        markersize=7,
        alpha=0.9,
        label=f'Rabi {label}'
    )

# PF
for i, label in enumerate([
    'Lower Polariton',
    'Upper Polariton',
    'Third Eigenstate'
]):
    ax1.plot(
        lambdas,
        PF_energies[:, i] * HF_TO_EV, color=state_colors[i], marker='d', linestyle='-.',
        linewidth=2.5,
        markersize=7,
        alpha=0.9,
        label=f'PF {label}'
    )

ax1.axhline(
    target_energy * HF_TO_EV, color='black', linestyle='-.',
    linewidth=2,
    alpha=0.7,
    label='Underlying Bright State'
)

ax1.set_xlabel('Coupling Strength (a.u.)')
ax1.set_ylabel('Energy (eV)')
ax1.set_title('Polariton Energies vs Coupling Strength')
ax1.legend(ncol=4, fontsize=8)
ax1.grid(alpha=0.3)

mask = lambdas >= 1e-8

# JC
for i, label in enumerate([
    'Lower Polariton',
    'Upper Polariton',
    'Third Polariton'
]):
    ax2.plot(
        lambdas[mask],
        JC_photon_contribution[:, i][mask], color=state_colors[i], marker='o', linestyle='-',
        linewidth=2,
        markersize=6,
        alpha=0.85,
        label=f'JC {label}'
    )

# RWA
for i, label in enumerate([
    'Lower Polariton',
    'Upper Polariton',
    'Third Polariton'
]):
    ax2.plot(
        lambdas[mask],
        RWA_photon_contribution[:, i][mask], color=state_colors[i], marker='s', linestyle='--',
        linewidth=2,
        markersize=6,
        alpha=0.85,
        label=f'RWA {label}'
    )

# Rabi
for i, label in enumerate([
    'Lower Polariton',
    'Upper Polariton',
    'Third Polariton'
]):
    ax2.plot(
        lambdas[mask],
        Rabi_photon_contribution[:, i][mask], color=state_colors[i], marker='^', linestyle=':',
        linewidth=2.5,
        markersize=7,
        alpha=0.9,
        label=f'Rabi {label}'
    )

# PF
for i, label in enumerate([
    'Lower Polariton',
    'Upper Polariton',
    'Third Polariton'
]):
    ax2.plot(
        lambdas[mask],
        PF_photon_contribution[:, i][mask], color=state_colors[i], marker='d', linestyle='-.',
        linewidth=2.5,
        markersize=7,
        alpha=0.9,
        label=f'PF {label}'
    )

ax2.set_xlabel('Coupling Strength (a.u.)')
ax2.set_ylabel('Photon Contribution')
ax2.set_title('Photon Character vs Coupling Strength')
ax2.legend(ncol=4, fontsize=8)
ax2.grid(alpha=0.3)

fig.tight_layout()
plt.show()

# ➡️ **Summary Table**

| **Model** | **DSE ($\Delta$)** | **CRTs** | **Complexity** | **Use case & Physical Regime** |
| --- | --- | --- | --- | --- |
| **JC** | No | No | Lowest | Weak coupling, quantum optics limit, $\lambda/\omega_c \ll 0.1$ | 
| **RWA** | Yes | No | Low | Moderate coupling, preserves ground state stability |
| **Rabi** | No | Yes | Medium | Strong/Ultra-strong coupling, but suffers from ground state collapse |
| **PF** | Yes | Yes | Highest | Ultra-strong & Deep strong coupling, ab initio accuracy, gauge-invariant |

> [!NOTE]
> **Gauge Invariance and Ground State Stability:**
> *   **DSE ($\Delta$)** acts as a restorative potential. Without it (as in JC and Rabi), the ground state has no lower bound at high coupling strengths, leading to a catastrophic collapse of polaritonic energies.
> *   **PF** is the only model that represents a fully self-consistent and gauge-invariant formulation of molecular cavity QED.

---
## 🏆 Part 5: Capstone Challenge – The Ultra-Strong Breakdown

Throughout this lesson, you have run individual calculations for the Jaynes-Cummings (JC), Rotating-Wave Approximation (RWA), and Rabi models. Now, we will visualize exactly why simple quantum optical models break down when light-matter interactions become extreme.

### The Task: 
Write a script that calculates the energy of the **Lower Polariton** using four cavity models (JC, RWA, Rabi, and PF) across a wide range of coupling strengths.

**1. Setup the Scan:**

Initialize Ethene and target its brightest transition, just as we did in Part 1. Set your coupling array to reach deep into the ultra-strong regime: `lambdas = np.linspace(0.0, 0.20, 20)`.

**2. The 4-Model Loop:**

Inside your loop over `lambdas`, dynamically align the cavity to the transition dipole. Then, initialize and solve all four QED-TDA models:
* `qed.JC`
* `qed.RWA`
* `qed.Rabi`
* `qed.PF`

Store the lowest polariton energy (index 0) for each model at every step.

**3. Visualizing the Breakdown:**

Plot all four arrays on the same graph: `Coupling Strength (a.u.)` on the x-axis, and `Lower Polariton Energy (eV)` on the y-axis. Use distinct markers and line styles (e.g. circles `o-` for JC, squares `s--` for RWA, triangles `^:` for Rabi, and diamonds `d-.` for PF) to make the comparison clear and professional.

### Analysis Questions:
* **The Symmetry Breaking:** At weak coupling ($\lambda < 0.05$), do the models agree? 
* **The Ground State Collapse:** Look at the JC and Rabi models (which both lack the Dipole Self-Energy term). What happens to their predicted energies as $\lambda$ approaches 0.20? 
* **The RWA Fix & Incompleteness:** How does the RWA model behave differently at high coupling, and why? How does it compare to the full Pauli-Fierz (PF) model?

In [ ]:
# 🏆 Capstone Challenge: Model Breakdown in the Ultra-Strong Regime
# Implement the scan up to lambda = 0.20 a.u. for JC, RWA, Rabi, and PF.

import numpy as np
import matplotlib.pyplot as plt
import qed

# 1. Setup the coupling strength array reaching deep into the USC regime (up to 0.20 au)
lambdas_cap = np.linspace(0.0, 0.20, 20)
cavity_freqs = np.array([target_energy])

# Storage lists for the Lower Polariton (LP) energy of each model
LP_JC = []
LP_RWA = []
LP_Rabi = []
LP_PF = []

# Unit vector along the targeted bright state transition dipole moment
unit_dip = trans_dip / np.linalg.norm(trans_dip)

print("Starting capstone calculations...")
for lam in lambdas_cap:
    print(f"Coupling strength lambda = {lam:.3f} a.u. ...", end="\r")
    
    # Define cavity mode vector aligned with transition dipole
    cavity_mode = (unit_dip * lam).reshape(3, 1)
    key_cap = {'cavity_mode': cavity_mode, 'cavity_freq': cavity_freqs}
    
    # 1. TDA-JC (Use nroots=5 to guarantee fast, stable convergence under degeneracy)
    cav_jc = qed.JC(mf, key_cap)
    td_jc = qed.TDA(mf, td, cav_jc, key_cap)
    td_jc.nroots = 5
    td_jc.kernel()
    LP_JC.append(td_jc.e[0] * HF_TO_EV)
    
    # 2. TDA-RWA (Use nroots=5 to guarantee stable convergence)
    cav_rwa = qed.RWA(mf, key_cap)
    td_rwa = qed.TDA(mf, td, cav_rwa, key_cap)
    td_rwa.nroots = 5
    td_rwa.kernel()
    LP_RWA.append(td_rwa.e[0] * HF_TO_EV)
    
    # 3. TDA-Rabi (Use nroots=5 to guarantee stable convergence)
    cav_rabi = qed.Rabi(mf, key_cap)
    td_rabi = qed.TDA(mf, td, cav_rabi, key_cap)
    td_rabi.nroots = 5
    td_rabi.kernel()
    LP_Rabi.append(td_rabi.e[0] * HF_TO_EV)
    
    # 4. TDA-PF (Use nroots=5 to guarantee stable convergence)
    cav_pf = qed.PF(mf, key_cap)
    td_pf = qed.TDA(mf, td, cav_pf, key_cap)
    td_pf.nroots = 5
    td_pf.kernel()
    LP_PF.append(td_pf.e[0] * HF_TO_EV)

print("\nCalculations completed successfully!")

# Convert to numpy arrays
LP_JC = np.array(LP_JC)
LP_RWA = np.array(LP_RWA)
LP_Rabi = np.array(LP_Rabi)
LP_PF = np.array(LP_PF)

# Plot the Lower Polariton Energy vs. Coupling Strength for all four models
plt.figure(figsize=(10, 6))
plt.plot(lambdas_cap, LP_JC, 'ro-', label='Jaynes-Cummings (JC)', markersize=6, linewidth=2)
plt.plot(lambdas_cap, LP_RWA, 'bs--', label='Rotating Wave Approx (RWA)', markersize=6, linewidth=2)
plt.plot(lambdas_cap, LP_Rabi, 'g^:', label='Rabi model', markersize=6, linewidth=2)
plt.plot(lambdas_cap, LP_PF, 'kd-.', label='Pauli-Fierz (PF)', markersize=6, linewidth=2)

plt.xlabel('Coupling Strength $\lambda$ (a.u.)', fontsize=12)
plt.ylabel('Lower Polariton Energy (eV)', fontsize=12)
plt.title('Collapse of Polariton Energies in the USC Regime', fontsize=14)
plt.legend(fontsize=10)
plt.grid(alpha=0.3)
plt.show()